# SEP and MTObjects MER VIS pilot
Run selected cells in order in the Euclid kernel. This uses three existing cutouts,
not full Q1 mosaics. Source files remain unchanged. Installation and results stay
under your Euclid_MER_VIS workspace. Review segmentation before adopting any masks.


In [ ]:
from pathlib import Path
import sys, subprocess, json
WORK = Path.cwd().resolve()
assert (WORK / 'MER_VIS_datalabs_manifest.csv').is_file(), 'Open this notebook in Euclid_MER_VIS'
assert (WORK / 'cutouts').is_dir(), 'Existing native cutouts are required'
ROOT = WORK / 'source_extraction_pilot'
print('Workspace:', WORK)
print('Pilot installation and results:', ROOT)


## Install inside the workspace
This downloads SEP and scikit image into a private environment that reuses the
Euclid scientific stack, and clones and compiles MTObjects using git and gcc.
No sudo or system package changes. Internet access is required for setup.
If MTObjects fails to build, SEP remains available and the installation record
will say why. Reruns reuse the existing MTObjects commit without updating it.


In [ ]:
import install_pilot
installation = install_pilot.install(ROOT)
PYTHON = ROOT / 'environment/bin/python'
print('MTObjects ready:', installation['mtobjects_ready'])


## Run three images
Defaults use at most a 1024 by 1024 native sampling crop centred on each galaxy.
These can be smaller than the original cutouts and may omit galaxy outskirts.
The first three suitable complete finite cutouts in manifest order are selected.
Each run gets a new results folder. MTObjects has a five minute timeout per image.
These are baseline feasibility settings, not your optimised local masking recipe.


In [ ]:
METHODS = ['sep', 'mto']  # Change to ['sep'] to run SEP alone.
command = [str(PYTHON), str(WORK / 'run_pilot.py'),
           '--work', str(WORK), '--root', str(ROOT),
           '--count', '3', '--max-side', '1024', '--methods', *METHODS]
with (ROOT / 'pilot_console.log').open('w') as log:
    result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
print((ROOT / 'pilot_console.log').read_text())
if result.returncode:
    raise RuntimeError('Pilot failed; inspect pilot_console.log')


## Display the comparison plots and statuses
Red crosses mark the catalogue centres. Green lines mark segmentation boundaries.
Detected regions can be stars, galaxies, companions or internal galaxy structure;
this pilot does not classify them as foreground or produce adopted masks.


In [ ]:
from IPython.display import display, Image
import csv
runs = sorted(ROOT.glob('results_*'))
assert runs, 'No pilot result folders'
LATEST = runs[-1]
print('Results:', LATEST)
with (LATEST / 'summary.csv').open(newline='') as f:
    for row in csv.DictReader(f): print(row)
for image in sorted(LATEST.glob('*/comparison.png')):
    display(Image(filename=str(image)))


## Return the small result files for review
Zip the latest results and download the ZIP using the file browser.
The results include cropped pilot inputs, labels, SEP background and RMS,
source tables, logs and comparison PNGs. No original full cutouts are packaged.


In [ ]:
import shutil
archive = shutil.make_archive(str(LATEST), 'zip', root_dir=LATEST)
print('Download:', archive)
